In [101]:
import duckdb
import polars as pl
import numpy as np
import pendulum
from pathlib import Path

data_dir = Path("../data/lastfm/listening")
db = duckdb.read_json(data_dir/"*.jsonl")
df = db.pl()

df = df.with_columns(
    pl.from_epoch(
        pl.col("date_played_unix"), time_unit="s"
    ).alias("track_played_utc")
)

print("Raw data:", df.shape)

df = df.unique("date_played_unix") # Not so exact with which duplicate gets removed
print("Duplicate timestamps removed:", df.unique("date_played_unix").shape)
df = df.with_columns(
    artist_name = pl.col("artist_name").str.to_lowercase(),
    track_name = pl.col("track_name").str.to_lowercase(),
    album_name = pl.col("album_name").str.to_lowercase()
)

Raw data: (110513, 8)
Duplicate timestamps removed: (108066, 8)


## Average session length (gap-based clustering)

In [102]:
year = 2025
df_gap = (
    df.sort(pl.col("date_played_unix"), descending=False)
    .filter(pl.col("track_played_utc").dt.year() == year)
    .with_columns(
        played_delta_s = pl.col("date_played_unix").diff().fill_null(0)
    )
).sort(pl.col("played_delta_s"), descending=True)
df_gap

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc,played_delta_s
str,str,str,str,str,str,i64,datetime[μs],i64
"""kettama""","""e457531c-4add-45f0-9998-f65ec5…","""yosemite""","""""","""yosemite""","""""",1735894188,2025-01-03 08:49:48,131901
"""astrid sonne""","""bc6af694-5880-4770-bdf1-82d03f…","""great doubt""","""df87374d-53df-48d9-9677-ae3297…","""say you love me""","""b38a04eb-3543-4286-add0-a81315…",1754286170,2025-08-04 05:42:50,107218
"""fcukers""","""5e23e428-d9f5-4679-bd7c-8254d6…","""bon bon""","""7758de6f-d039-4672-ac4b-2c30fc…","""bon bon""","""864c5765-1c94-489a-80d7-7557de…",1752418527,2025-07-13 14:55:27,100913
"""fred again..""","""bca46a0c-25c9-42ca-98c2-e64c8a…","""actual life 2 (february 2 - oc…","""f50f4e97-8a6d-4212-a710-e82caf…","""roze (forgive)""","""9a3467f6-4fd8-45f8-a855-e552c0…",1766700505,2025-12-25 22:08:25,83148
"""yung lean""","""""","""forever yung""","""""","""forever yung""","""""",1740330680,2025-02-23 17:11:20,81702
…,…,…,…,…,…,…,…,…
"""caroline polachek""","""d8f43dc5-4613-48ac-8c23-a62b82…","""pang""","""3913f9b2-8622-4b0b-aeb5-3511d2…","""i give up""","""552cf74f-2de3-46c5-8adb-3955a0…",1765965150,2025-12-17 09:52:30,1
"""lover's skit""","""""","""all rights remixed""","""""","""no te metas - sassy 009 remix""","""""",1765965328,2025-12-17 09:55:28,1
"""amelie lens""","""67db280d-c3e4-49d9-978e-4050f4…","""higher ep""","""ab5756fb-afda-434e-906a-4fbca0…","""higher - fjaak remix""","""""",1766316371,2025-12-21 11:26:11,1


In [103]:
custom_bins = np.arange(0, 86400+1, 600)
bin_count, bins = np.histogram(
    df_gap.select(pl.col("played_delta_s")).to_numpy().flatten(),
    # 200
    custom_bins
)
print(bin_count)
print(bins)

[16843   791   389   256   216   175   148   100    89    76    63    54
    50    39    30    41    39    29    28    27    19    20    17    20
    13    11    15     9     8     7     7     2     3     2     4     2
     3     2     6     3     5     3     3     3     3     0     2     3
     2     8     7     4     2     3     4     7     7     3     7     9
     9    11    16     7    11     9    13    13    14    17     9     7
     5     8     8    14     7    10    10     6     5     1     6     8
     4     4    12     4     2     3     2     7     3     1     4     3
     2     1     2     2     2     1     2     2     1     1     0     0
     1     4     1     0     1     0     0     1     1     0     1     0
     1     0     0     0     0     0     1     0     0     0     0     0
     0     0     1     1     1     0     1     0     0     0     0     0]
[    0   600  1200  1800  2400  3000  3600  4200  4800  5400  6000  6600
  7200  7800  8400  9000  9600 10200 10800 11400 1

We will set 10 minutes (600s) as our threshold for what consitutes as a new listening session.

In [104]:
threshold = 1800 # 3600
# threshold = 3600 # 3600
df_sessions = (
    df_gap
    .sort(pl.col("date_played_unix"), descending=False)
    .with_columns(
        (pl.col("played_delta_s") > threshold).alias("exceeds")
    )
    .with_columns(pl.col("exceeds").cum_sum().alias("nth_session"))
    .drop("exceeds")
    # count
    # .group_by(pl.col("track_played_utc"), pl.col("nth_session"))
    # .agg(
    #     n_scrobbles = pl.len()
    # )
    # .sort(pl.col("track_played_utc"), descending=False)
    # .with_columns(
    #     nth_scrobble = pl.col("n_scrobbles").cum_sum()
    # )
)
df_sessions.head(10)

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc,played_delta_s,nth_session
str,str,str,str,str,str,i64,datetime[μs],i64,u32
"""brutalismus 3000""","""b4cd749e-adce-4cb5-8a73-cdd0c7…","""goodbye salò""","""""","""europaträume""","""a8902a2d-906d-4595-ac0e-68459c…",1735693667,2025-01-01 01:07:47,0,0
"""mutilator""","""b239fbf8-782b-4dc0-85eb-3ff4d6…","""can you feel it?!""","""""","""can you feel it?!""","""""",1735693827,2025-01-01 01:10:27,160,0
"""snow strippers""","""33243fef-ce1a-4ba7-a81e-ec72e4…","""april mixtape 2""","""3c39c467-3e37-4a41-b530-b68429…","""we both suffocate""","""27a98bf0-a2fb-4b67-bb69-84fc52…",1735694113,2025-01-01 01:15:13,286,0
"""underscores""","""ab1a3f85-e0ea-470a-af5c-175447…","""wallsocket (director's cut)""","""06efd7a6-f6ee-406d-8575-2c52a0…","""stupid (can’t run from the urg…","""404bd1ba-e91a-401c-b4b3-1626ce…",1735694276,2025-01-01 01:17:56,163,0
"""smino""","""""","""maybe in nirvana""","""""","""hoe-nouns (feat. thundercat & …","""""",1735754288,2025-01-01 17:58:08,60012,1
"""saga bouy""","""""","""scary""","""""","""scary""","""""",1735755813,2025-01-01 18:23:33,1525,1
"""saga bouy""","""""","""scary""","""""","""scary""","""""",1735762287,2025-01-01 20:11:27,6474,2
"""kettama""","""e457531c-4add-45f0-9998-f65ec5…","""yosemite""","""""","""yosemite""","""""",1735894188,2025-01-03 08:49:48,131901,3
"""eliza rose""","""afd9baa6-4402-40c8-91ad-d792ae…","""b.o.t.a. (baddest of them all)…","""""","""b.o.t.a. (baddest of them all)""","""""",1735894453,2025-01-03 08:54:13,265,3


### Session statistics

In [105]:
df_session_stats = ( 
    df_sessions 
    .group_by(pl.col("nth_session"))
    .agg(
        session_mean_s = pl.col("played_delta_s").mean(),
        tracks_played = pl.col("played_delta_s").count(),
        session_start = pl.col("track_played_utc").min(),
        session_end = pl.col("track_played_utc").max(),
    )
    .with_columns(
        session_length = (pl.col("session_end") - pl.col("session_start"))# .dt.total_seconds()
    )
    .filter(pl.col("session_length") > pl.duration(hours=3))
).sort(pl.col("session_start"))
df_session_stats.sort(pl.col("session_length"), descending=True)

nth_session,session_mean_s,tracks_played,session_start,session_end,session_length
u32,f64,u32,datetime[μs],datetime[μs],duration[μs]
914,332.760479,167,2025-06-12 06:09:46,2025-06-12 16:11:29,10h 1m 43s
58,233.576923,130,2025-01-13 07:00:56,2025-01-13 14:06:24,7h 5m 28s
311,290.715909,88,2025-02-27 07:24:08,2025-02-27 13:39:04,6h 14m 56s
213,266.295918,98,2025-02-07 06:44:35,2025-02-07 12:57:16,6h 12m 41s
354,238.768519,108,2025-03-07 07:10:54,2025-03-07 13:18:16,6h 7m 22s
…,…,…,…,…,…
963,508.75,44,2025-06-19 14:44:15,2025-06-19 17:48:34,3h 4m 19s
719,1032.489362,47,2025-05-12 06:51:30,2025-05-12 09:54:30,3h 3m
1586,311.634146,41,2025-10-10 07:18:27,2025-10-10 10:20:45,3h 2m 18s


In [106]:
import plotly.express as px
fig = px.timeline(df_session_stats, x_start="session_start", x_end="session_end", color="session_length")
fig.show()

In [109]:
# Average session length
df_session_stats.select(pl.col("session_length").mean().alias("avg_session_length"))

avg_session_length
duration[μs]
4h 16m 57s 150684µs
